# 02. Поверхности контекста

Контекст это не только «последние сообщения чата». В агенте есть несколько поверхностей, которые могут попасть в модель: инструкции, задача, стейт, результаты инструментов, retrieval, память, схемы вывода.

До кэпстоуна используем нейтральный docs-like сценарий: служба поддержки, заказ, возврат, FAQ и простые действия вроде проверки статуса.

В этом примере мы заведем маленькую таблицу и посмотрим, какие части контекста можно считать доверенными.

In [ ]:
surfaces = [
    {
        "surface": "developer_instructions",
        "example": "Не записывать wiki-страницы без preview.",
        "trust": "high",
        "risk": "Если правило потерять, агент начнет делать silent overwrite.",
    },
    {
        "surface": "user_task",
        "example": "Проверь статус заказа ORD-1842.",
        "trust": "medium",
        "risk": "Пользователь может просить слишком широкое действие.",
    },
    {
        "surface": "retrieved_source",
        "example": "Фрагмент postmortem из базы источников.",
        "trust": "untrusted",
        "risk": "В документе может быть prompt injection или устаревший факт.",
    },
    {
        "surface": "tool_result",
        "example": "Список файлов в vault/wiki.",
        "trust": "tool_bounded",
        "risk": "Результат надо валидировать и не смешивать с инструкциями.",
    },
    {
        "surface": "output_schema",
        "example": "JSON с полями action, page, reason, citations.",
        "trust": "high",
        "risk": "Без схемы сложнее проверить ответ программы.",
    },
]

for item in surfaces:
    print(f"{item['surface']}: {item['trust']}")
    print(f"  пример: {item['example']}")
    print(f"  риск: {item['risk']}\n")

Практическое правило: **недоверенный текст не должен становиться инструкцией**. Его можно цитировать, сравнивать, проверять и использовать как данные, но нельзя позволять ему управлять харнесом.

In [ ]:
def group_by_trust(items):
    groups = {}
    for item in items:
        groups.setdefault(item["trust"], []).append(item["surface"])
    return groups


group_by_trust(surfaces)